# 📝 Day 7 Assignments — SQL & SQLAlchemy

---

Four tasks plus a bonus. You'll write the same logic twice — first with raw `sqlite3`, then with SQLAlchemy 2.0 — so the ORM feels like a natural step up rather than magic.

In [ ]:
!pip install sqlalchemy

## Task 1 — Raw `sqlite3`: a `students` table

**Problem:** Using the stdlib `sqlite3` module:

1. Open an in-memory connection.
2. Create a `students` table with columns `id` (PK), `name` (TEXT), `age` (INTEGER), `marks` (INTEGER).
3. Insert 3 rows.
4. `SELECT *` and print each row.

**Expected output:**
```
(1, 'Alice', 20, 88)
(2, 'Bob', 17, 72)
(3, 'Carol', 22, 95)
```

💡 **Hint:** Use `?` placeholders, not f-strings. Don't forget `conn.commit()`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# TODO: CREATE TABLE students (...)

# TODO: INSERT 3 rows with executemany or three execute() calls

# TODO: SELECT * and print every row

conn.close()

## Task 2 — Same thing with SQLAlchemy 2.0 ORM

**Problem:** Redo Task 1 with the ORM:

1. Define `Student(Base)` with the same columns using `Mapped` / `mapped_column`.
2. `create_all` against `sqlite:///./day7_assignments.db`.
3. Insert 3 students via a session.
4. `select(Student)` and print each one.

**Expected output:** three `Student(...)` objects.

💡 **Hint:** `id: Mapped[int] = mapped_column(primary_key=True)` gives you autoincrement for free.

In [ ]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker

engine = create_engine("sqlite:///./day7_assignments.db", echo=False)

class Base(DeclarativeBase):
    pass

class Student(Base):
    __tablename__ = "students"
    # TODO: id (PK), name, age, marks

    def __repr__(self) -> str:
        return f"Student(id={self.id}, name={self.name!r}, age={self.age}, marks={self.marks})"

# TODO: create_all

SessionLocal = sessionmaker(bind=engine)

with SessionLocal() as session:
    # TODO: clear the table so re-runs are idempotent (hint: from sqlalchemy import delete)
    # TODO: add 3 Student rows
    # TODO: select(Student) and print each
    pass

## Task 3 — `WHERE` queries

**Problem:** Find every student with `age > 18`, twice:

1. With raw `sqlite3` (`SELECT ... WHERE age > ?`).
2. With SQLAlchemy (`select(Student).where(Student.age > 18)`).

Print both result sets and confirm they match.

💡 **Hint:** In SQLAlchemy use `Student.age > 18` (a Python expression — it produces a SQL clause).

In [ ]:
# Raw sqlite3 version — reuse a fresh in-memory DB and seed the same 3 rows
# TODO

# SQLAlchemy version — reuse `engine` / `Student` from Task 2
# TODO: print students where age > 18

## Task 4 — `UPDATE` and `DELETE`

**Problem:** Using the SQLAlchemy `Student` table from Task 2:

1. Update one student's name.
2. Delete a different student.
3. `select(Student)` to verify the final state.

💡 **Hint:** Mutate the loaded object and `session.commit()` — no `UPDATE` string needed.

In [ ]:
with SessionLocal() as session:
    # TODO: rename one student
    # TODO: delete another student
    # TODO: print all remaining rows
    pass

## 🎁 Bonus — Indexes and `unique=True`

**Problem:**

1. Add an `email` column to `Student`:
   ```python
   email: Mapped[str] = mapped_column(index=True, unique=True)
   ```
   (You'll need a fresh DB file — schema changes aren't automatic without migrations, which we'll cover Day 8.)
2. Query a student **by email**.
3. In a markdown cell below, explain in 2–3 sentences why this matters when the table grows to **1,000,000 rows**.

💡 **Hint:** without an index, `WHERE email = ?` is a full table scan — O(n). With an index it's roughly O(log n).

In [ ]:
# Use a NEW db file so the schema change is picked up cleanly
engine2 = create_engine("sqlite:///./day7_bonus.db", echo=False)

class Base2(DeclarativeBase):
    pass

class StudentV2(Base2):
    __tablename__ = "students"
    id:    Mapped[int] = mapped_column(primary_key=True)
    name:  Mapped[str]
    age:   Mapped[int]
    marks: Mapped[int]
    # TODO: email: Mapped[str] = mapped_column(index=True, unique=True)

Base2.metadata.create_all(engine2)
Session2 = sessionmaker(bind=engine2)

with Session2() as session:
    # TODO: insert a couple of students with emails
    # TODO: query by email and print the result
    pass

### Why indexes matter — your answer here

> _Write 2–3 sentences explaining what would happen if you `WHERE email = ?` against a 1M-row table without an index._

---

✅ **Done!** You can now drive SQLite both raw and through SQLAlchemy 2.0. Tomorrow we add relationships and plug all of this into FastAPI.